step 1 ==> in this notebook we are initializing our project, like extracting data from its source(reading raw data), adding meta data, and 
storing raw data into bronze layer in the form of (delta data (.parquet))


In [0]:
customer_ds_path="/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_customers_dataset.csv"
order_item_ds_path = "/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_order_items_dataset.csv"
orders_ds_path="/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_orders_dataset.csv"
product_ds_path="/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_products_dataset.csv"

In [0]:
customers_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(customer_ds_path)
)

orders_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(orders_ds_path)
)

order_items_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(order_item_ds_path)
)

products_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(product_ds_path)
)


In [0]:
customers_df.count()

99441

In [0]:
orders_df.count()

99441

In [0]:
order_items_df.count()


112650

In [0]:

products_df.count()

32951

In [0]:
from pyspark.sql.functions import col, current_timestamp

customers_bronze_df = (
    customers_df
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("ingestion_timestamp", current_timestamp())
)

orders_bronze_df = (
    orders_df
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("ingestion_timestamp", current_timestamp())
)

order_items_bronze_df = (
    order_items_df
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("ingestion_timestamp", current_timestamp())
)

products_bronze_df = (
    products_df
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("ingestion_timestamp", current_timestamp())
)

In [0]:
customers_bronze_df.limit(10).display()

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,source_file,ingestion_timestamp
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_customers_dataset.csv,2026-09-20T10:29:45.159Z
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_customers_dataset.csv,2026-09-20T10:29:45.159Z
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_customers_dataset.csv,2026-09-20T10:29:45.159Z
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_customers_dataset.csv,2026-09-20T10:29:45.159Z
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_customers_dataset.csv,2026-09-20T10:29:45.159Z
879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_customers_dataset.csv,2026-09-20T10:29:45.159Z
fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,4534,sao paulo,SP,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_customers_dataset.csv,2026-09-20T10:29:45.159Z
5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,timoteo,MG,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_customers_dataset.csv,2026-09-20T10:29:45.159Z
5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_customers_dataset.csv,2026-09-20T10:29:45.159Z
4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,belo horizonte,MG,dbfs:/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Raw/olist_customers_dataset.csv,2026-09-20T10:29:45.159Z


In [0]:
bronze_layer_path = "/Volumes/brazilian_e-commerce_olist_project/default/e-commerce_datasets/Bronze/"

(customers_bronze_df.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .save(f"{bronze_layer_path}/customers")
)

(orders_bronze_df.write 
    .format("delta") 
    .mode("overwrite") 
    .save(f"{bronze_layer_path}/orders")
)

(order_items_bronze_df.write 
    .format("delta") 
    .mode("overwrite") 
    .save(f"{bronze_layer_path}/order_items")
)

(products_bronze_df.write 
    .format("delta") 
    .mode("overwrite") 
    .save(f"{bronze_layer_path}/products"))